In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [5]:
!wget

wget: missing URL
Usage: wget [OPTION]... [URL]...

Try `wget --help' for more options.


In [6]:
df = pd.read_csv(data)
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [7]:
df.isnull().sum() #checking missing values in the data set

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [8]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [9]:
(df.converted=='0').head()

0    False
1    False
2    False
3    False
4    False
Name: converted, dtype: bool

In [10]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

In [11]:
df[['annual_income','lead_score']].fillna(0)

,annual_income,lead_score
0,88160.0,0.64
1,72688.0,0.72
2,44697.0,0.58
3,0.0,0.57
4,24062.0,0.62
...,...,...
4995,45834.0,0.37
4996,84831.0,0.51
4997,53636.0,0.75
4998,69430.0,0.79


In [12]:
df[['lead_source','industry','employment_status','location']].fillna("NA")

,lead_source,industry,employment_status,location
0,organic_search,technology,employed,europe
1,social_media,technology,employed,south_america
2,referral,retail,employed,europe
3,paid_ads,manufacturing,student,NA
4,referral,manufacturing,student,north_america
...,...,...,...,...
4995,organic_search,education,employed,asia
4996,NA,technology,employed,north_america
4997,organic_search,finance,employed,asia
4998,referral,technology,employed,africa


In [13]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [14]:
df_corr_matrix = df.corr(numeric_only =True)
abs_corr = df_corr_matrix.abs().unstack().drop_duplicates() # dropping the self correlation (in case a variable is pairing with itself
biggest_corr = abs_corr[abs_corr < 1.0].sort_values(ascending = False) #getting the biggest correlations from variables pairing 
biggest_corr


interaction_count         lead_score                  0.928712
number_of_courses_viewed  lead_score                  0.769352
                          interaction_count           0.721609
lead_score                converted                   0.490206
interaction_count         converted                   0.448624
number_of_courses_viewed  converted                   0.395038
annual_income             lead_score                  0.231396
                          converted                   0.181752
                          number_of_courses_viewed    0.137542
                          interaction_count           0.106375
dtype: float64

In [26]:
# splitting data into train/trai data
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size = 0.2, random_state = 1 )
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 42)
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [27]:
#reset index

df_train = df_train.reset_index(drop = True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop = True)


In [31]:
# Isolating the target(converted) to avoid training it with the entire data by extracting it as a numpy array with .values
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values


In [34]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [38]:
# for the exploratory data analysis

df_full_train.reset_index(drop =True)
df_full_train.isnull().sum()

lead_source                 119
industry                    200
employment_status           156
location                    167
annual_income               300
number_of_courses_viewed      0
interaction_count             0
lead_score                   30
converted                     0
dtype: int64

In [40]:

#Ensuring no missing values. I filled categorical data with NA and the numerical data with 9
df_full_train[['lead_source','industry','employment_status','location']].fillna('NA')# Categorical features

,lead_source,industry,employment_status,location
1233,paid_ads,technology,student,europe
1056,paid_ads,healthcare,self_employed,north_america
1686,paid_ads,retail,employed,NA
187,social_media,retail,self_employed,NA
3840,organic_search,retail,student,north_america
...,...,...,...,...
2895,paid_ads,manufacturing,NA,north_america
2763,paid_ads,manufacturing,employed,north_america
905,NA,manufacturing,NA,NA
3980,organic_search,manufacturing,self_employed,north_america


In [46]:
df_full_train[['annual_income','lead_score']].fillna(0) # numerical features

,annual_income,lead_score
1233,38757.0,0.15
1056,56516.0,0.59
1686,54963.0,0.13
187,26237.0,0.46
3840,21992.0,0.70
...,...,...
2895,66121.0,0.20
2763,60720.0,0.69
905,49820.0,0.44
3980,52228.0,0.50


In [47]:
df_full_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
1233,paid_ads,technology,student,europe,38757.0,0,0,0.15,0
1056,paid_ads,healthcare,self_employed,north_america,56516.0,2,6,0.59,1
1686,paid_ads,retail,employed,NaN,54963.0,1,0,0.13,0
187,social_media,retail,self_employed,NaN,26237.0,1,5,0.46,0
3840,organic_search,retail,student,north_america,21992.0,3,6,0.70,1
...,...,...,...,...,...,...,...,...,...
2895,paid_ads,manufacturing,NaN,north_america,66121.0,1,1,0.20,0
2763,paid_ads,manufacturing,employed,north_america,60720.0,4,6,0.69,1
905,NaN,manufacturing,NaN,NaN,49820.0,2,3,0.44,0
3980,organic_search,manufacturing,self_employed,north_america,52228.0,3,5,0.50,0


In [48]:
#looking at the target variables. we check how many customers converted and those not
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57875
0    0.42125
Name: proportion, dtype: float64

In [49]:
# 58.8% where converted while 42% where not
df_full_train.converted.mean() # Calculating conversion rate 

np.float64(0.57875)

In [53]:
# Getting unique values from each feature using numeriacl and cateogical values

numerical = ['annual_income','number_of_courses_viewed', 'interaction_count','lead_score']
categorical =['lead_source','industry','employment_status','location']
df_full_train[categorical].nunique()

lead_source          5
industry             6
employment_status    4
location             5
dtype: int64